# 10 Machine Learning: Predicting Infection and Severe Outcomes

Can we predict who will get infected and who will become severe from residents' basic information?

Workflow: **problem definition → feature engineering → Pipeline → cross-validation AUC → Random Forest → feature importance → model comparison**

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: Problem definition and target variables ---
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# Task A: predict infection
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# Task B: predict severe outcome (hospitalized or died)
df["severe_outcome"] = (
    (df["hospitalized"] == 1) | (df["outcome"] == "dead")
).astype(int)

print("=== Distribution of prediction targets ===")
print(f"Task A (infected):       {df['infected'].value_counts().to_dict()}")
print(f"Task B (severe_outcome): {df['severe_outcome'].value_counts().to_dict()}")
print(f"\nTask A positive-class share: {df['infected'].mean():.1%}")
print(f"Task B positive-class share: {df['severe_outcome'].mean():.1%}")

In [ ]:
# --- Step 2: Feature engineering ---
# Note: we must not use symptoms (fever, cough...) as features!
# Symptoms only appear *after* infection -> data leakage

num_cols = ["age"]  # Numeric features

cat_cols = ["sex", "smoking_history", "functional_status", "wing"]  # Categorical features

bin_cols = [  # Binary features (0/1)
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]

feature_cols = num_cols + cat_cols + bin_cols
X = df[feature_cols]
y_infected = df["infected"]
y_severe = df["severe_outcome"]

print(f"Number of features: {len(feature_cols)}")
print(f"  Numeric:     {num_cols}")
print(f"  Categorical: {cat_cols}")
print(f"  Binary:      {bin_cols}")
print(f"\nNumber of samples: {len(X)}")

In [ ]:
# --- Step 3: sklearn Pipeline ---
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Preprocessing: scale numeric + one-hot categorical + pass binary through directly
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    ("bin", "passthrough", bin_cols),
])

# Logistic Regression baseline
clf_lr = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=500, random_state=42)),
])

print("Pipeline structure:")
print(clf_lr)

In [ ]:
# --- Step 4: Cross-validation + AUC ---
# Task A: predict infection
scores_lr_a = cross_val_score(clf_lr, X, y_infected, cv=5, scoring="roc_auc")
print(f"=== Task A: predict infection ===")
print(f"Logistic Regression 5-fold CV AUC = {scores_lr_a.mean():.3f} ± {scores_lr_a.std():.3f}")

# Task B: predict severe outcome
scores_lr_b = cross_val_score(clf_lr, X, y_severe, cv=5, scoring="roc_auc")
print(f"\n=== Task B: predict severe outcome ===")
print(f"Logistic Regression 5-fold CV AUC = {scores_lr_b.mean():.3f} ± {scores_lr_b.std():.3f}")

print("\n→ AUC 0.5 = random guessing, 0.7+ = acceptable, 0.8+ = good")

In [ ]:
# --- Step 5: Random Forest, a more advanced model ---
from sklearn.ensemble import RandomForestClassifier

clf_rf = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
])

# Task A
scores_rf_a = cross_val_score(clf_rf, X, y_infected, cv=5, scoring="roc_auc")
print(f"=== Task A: predict infection ===")
print(f"Random Forest 5-fold CV AUC = {scores_rf_a.mean():.3f} ± {scores_rf_a.std():.3f}")

# Task B
scores_rf_b = cross_val_score(clf_rf, X, y_severe, cv=5, scoring="roc_auc")
print(f"\n=== Task B: predict severe outcome ===")
print(f"Random Forest 5-fold CV AUC = {scores_rf_b.mean():.3f} ± {scores_rf_b.std():.3f}")

In [ ]:
# --- Step 6: Feature importance (permutation importance) ---
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_infected, test_size=0.3, random_state=42,
)

clf_rf.fit(X_train, y_train)
perm = permutation_importance(
    clf_rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc",
)

# Sort
imp_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_df["feature"], imp_df["importance"], xerr=imp_df["std"],
        color="#2c7fb8", alpha=0.8)
ax.set_xlabel("Permutation Importance (AUC decrease)")
ax.set_title("Task A (infected) — Feature Importance")
plt.tight_layout()
plt.show()

print("\n=== Top 5 most important features ===")
top5 = imp_df.nlargest(5, "importance")
for _, row in top5.iterrows():
    print(f"  {row['feature']:25s}  importance = {row['importance']:.4f}")

In [ ]:
# --- Step 7: Model comparison summary ---
from sklearn.metrics import roc_auc_score

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Task A AUC": [
        f"{scores_lr_a.mean():.3f} ± {scores_lr_a.std():.3f}",
        f"{scores_rf_a.mean():.3f} ± {scores_rf_a.std():.3f}",
    ],
    "Task B AUC": [
        f"{scores_lr_b.mean():.3f} ± {scores_lr_b.std():.3f}",
        f"{scores_rf_b.mean():.3f} ± {scores_rf_b.std():.3f}",
    ],
})
print("=== Model comparison ===")
print(results.to_string(index=False))

print("\n→ On a small sample (280 rows), a simple Logistic Regression usually performs about as well as a Random Forest")
print("→ Complex models overfit easily; the cross-validation standard deviation reflects this")
print("→ Does the ML feature importance point in the same direction as the adjusted OR from the Ch06 logistic regression?")

## Summary

| Step | Skill learned |
|------|------------|
| Problem definition | Distinguish Task A (infection) vs Task B (severe outcome), avoid data leakage |
| Feature engineering | Handle numeric / categorical / binary features separately |
| Pipeline | `ColumnTransformer` + `Pipeline` keep preprocessing inside each CV fold |
| Cross-validation | `cross_val_score(cv=5, scoring='roc_auc')` |
| Random Forest | A nonlinear model, but with limited advantage on small samples |
| Feature importance | `permutation_importance` finds the most predictive features |

**Conclusion**: An ML model on 280 rows can serve as a baseline, but its predictive performance is limited by the sample size.
What matters is the ranking of feature importance—if ML and regression analysis point to the same risk factors, the conclusion is more trustworthy.

In the next chapter (Ch11), we'll try deep learning with PyTorch—and discuss whether it makes sense to use DL on just 280 rows.